In [14]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def crawl_naver_search_exact_url_full(target_url):
    # 크롬 드라이버 설정
    options = webdriver.ChromeOptions()
    options.add_argument('--window-size=1920,1080')
    # 백그라운드에서 실행하려면 아래 주석 해제
    # options.add_argument('--headless') 
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    
    driver.get(target_url)
    time.sleep(3)
    
    post_links = set()
    print("전체 링크 수집을 위한 무한 스크롤을 시작합니다...")
    
    # 1. 무한 스크롤 및 전체 링크 수집
    last_height = driver.execute_script("return document.body.scrollHeight")
    scroll_attempts = 0
    max_scroll_attempts = 3 # 3회 연속 스크롤 높이 변화가 없으면 종료
    
    while True:
        # 화면에 로딩된 블로그 링크 추출
        elements = driver.find_elements(By.CSS_SELECTOR, "a[href*='blog.naver.com']")
        
        for el in elements:
            href = el.get_attribute("href")
            if href and "blog.naver.com" in href:
                # 파라미터 제거 및 순수 게시글 URL 추출
                clean_href = href.split('?')[0].rstrip('/')
                parts = clean_href.split('/')
                
                # 끝자리가 숫자인 실제 블로그 포스팅 링크만 선별
                if len(parts) >= 4 and parts[-1].isdigit():
                    post_links.add(clean_href)
        
        # 스크롤 다운
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.5) # 로딩 대기
        
        # 스크롤 높이 비교로 마지막 페이지 도달 여부 확인
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            scroll_attempts += 1
            if scroll_attempts >= max_scroll_attempts:
                print("더 이상 불러올 게시글이 없습니다. 스크롤을 종료합니다.")
                break
            time.sleep(1) # 일시적인 로딩 지연 대비 추가 대기
        else:
            last_height = new_height
            scroll_attempts = 0 # 높이가 변했다면 시도 횟수 초기화
            
    post_links = list(post_links)
    print(f"\n총 {len(post_links)}개의 블로그 링크 추출 완료. 상세 데이터 수집을 시작합니다.")
    print("데이터 양에 따라 다소 시간이 소요될 수 있습니다.\n")
    
    # 2. 각 링크에 접속하여 상세 데이터 추출
    final_data_list = []
    
    for idx, link in enumerate(post_links, 1):
        try:
            driver.switch_to.default_content()
            driver.get(link)
            time.sleep(2.5) # 대량 수집 시 차단 방지를 위한 안정적인 딜레이
            
            # 블로그 본문 iframe 진입
            WebDriverWait(driver, 10).until(EC.frame_to_be_available_and_switch_to_it((By.NAME, "mainFrame")))
            
            html = driver.page_source
            soup = BeautifulSoup(html, 'html.parser')
            
            blogger_id = link.split('/')[-2]
            post_date = ""
            post_text = ""
            like_count = "0"
            comment_count = "0"
            tags = ""
            
            # 날짜 수집
            date_tag = soup.select_one('.se_publishDate') or soup.select_one('.blog2_date')
            if date_tag:
                post_date = date_tag.text.strip()
                
            # 본문 수집
            content_tag = soup.select_one('.se-main-container') or soup.select_one('#postViewArea')
            if content_tag:
                post_text = content_tag.get_text(separator=' ', strip=True)
                
            # 태그 수집
            tag_elements = soup.select('.item_tag') or soup.select('.wrap_tag > a')
            tags = ", ".join([tag.text.strip().replace('#', '') for tag in tag_elements])
            
            # 좋아요 수 수집
            try:
                WebDriverWait(driver, 3).until(EC.presence_of_element_located((By.CSS_SELECTOR, "em.u_cnt._count")))
                like_count_element = driver.find_element(By.CSS_SELECTOR, "em.u_cnt._count")
                if like_count_element.text:
                    like_count = like_count_element.text
            except:
                pass
                
            # 댓글 수 수집
            try:
                comment_count_element = driver.find_element(By.CSS_SELECTOR, "em#commentCount")
                if comment_count_element.text:
                    comment_count = comment_count_element.text
            except:
                pass
                
            final_data_list.append({
                "아이디": blogger_id,
                "날짜": post_date,
                "본문 텍스트": post_text,
                "좋아요 수": like_count,
                "댓글 수": comment_count,
                "태그": tags
            })
            
            # 진행 상황 모니터링
            print(f"[{idx}/{len(post_links)}] 수집 완료 (작성일: {post_date})")
            
        except Exception as e:
            print(f"[{idx}/{len(post_links)}] 수집 에러 발생 ({link}): {e}")
            continue
            
    driver.quit()
    return pd.DataFrame(final_data_list)

if __name__ == "__main__":
    TARGET_URL = "https://search.naver.com/search.naver?ssc=tab.blog.all&sm=tab_jum&query=%22%EC%97%90%EB%B2%84%EB%9E%9C%EB%93%9C%22+%22%EB%B0%A9%EB%AC%B8%EA%B8%B0%22&nso=p%3Afrom20240101to20261022"
    
    result_df = crawl_naver_search_exact_url_full(target_url=TARGET_URL)
    
    file_name = "naver_blog_reviews_full_collection.csv"
    result_df.to_csv(file_name, index=False, encoding='utf-8-sig')
    print(f"\n최종 수집 완료! 총 {len(result_df)}개의 데이터가 '{file_name}' 파일로 저장되었습니다.")

전체 링크 수집을 위한 무한 스크롤을 시작합니다...
더 이상 불러올 게시글이 없습니다. 스크롤을 종료합니다.

총 980개의 블로그 링크 추출 완료. 상세 데이터 수집을 시작합니다.
데이터 양에 따라 다소 시간이 소요될 수 있습니다.

[1/980] 수집 완료 (작성일: 2024. 3. 4. 8:00)
[2/980] 수집 완료 (작성일: 2026. 8. 25. 15:32)
[3/980] 수집 완료 (작성일: 2025. 10. 22. 18:22)
[4/980] 수집 완료 (작성일: 2025. 11. 2. 22:08)
[5/980] 수집 완료 (작성일: 2024. 10. 20. 18:03)
[6/980] 수집 완료 (작성일: 2026. 7. 12. 8:38)
[7/980] 수집 완료 (작성일: 2026. 10. 15. 16:55)
[8/980] 수집 완료 (작성일: 2025. 10. 10. 12:10)
[9/980] 수집 완료 (작성일: 2026. 7. 20. 11:14)
[10/980] 수집 완료 (작성일: 2026. 5. 17. 18:30)
[11/980] 수집 완료 (작성일: 2026. 5. 13. 22:37)
[12/980] 수집 완료 (작성일: 2025. 5. 30. 3:40)
[13/980] 수집 완료 (작성일: 2026. 2. 18. 22:48)
[14/980] 수집 완료 (작성일: 2024. 10. 5. 0:12)
[15/980] 수집 완료 (작성일: 2024. 8. 17. 18:04)
[16/980] 수집 완료 (작성일: 2026. 7. 22. 11:00)
[17/980] 수집 완료 (작성일: 2026. 7. 15. 21:04)
[18/980] 수집 완료 (작성일: 2026. 7. 8. 15:17)
[19/980] 수집 완료 (작성일: 2024. 9. 25. 10:45)
[20/980] 수집 완료 (작성일: 2024. 11. 6. 19:56)
[21/980] 수집 완료 (작성일: 2026. 9. 20. 10:29)
[22/980] 수집 완료 (작

In [ ]:
import pandas as pd
import re

# 1. 수집된 파일 불러오기
df = pd.read_csv('blog_post.csv')

# 2. 본문 텍스트 내의 해시태그 분리 및 제거 함수
def clean_text_and_extract_tags(text):
    if not isinstance(text, str):
        return text, ""
    
    # 본문에 포함된 #태그 형태의 단어 모두 추출
    inline_tags = re.findall(r'#([^\s#]+)', text)
    inline_tags_str = ", ".join(inline_tags)
    
    # 본문에서는 #태그 단어들을 모두 삭제하고 공백 정리
    clean_text = re.sub(r'#[^\s#]+', '', text).strip()
    # 띄어쓰기가 여러 개 겹친 곳을 하나로 정리
    clean_text = re.sub(r'\s+', ' ', clean_text)
    
    return clean_text, inline_tags_str

# 3. 데이터프레임에 적용
df[['정제된 본문', '본문 내 태그']] = df['본문 텍스트'].apply(lambda x: pd.Series(clean_text_and_extract_tags(x)))

# (선택) 기존 태그 열과 본문 내 태그 열을 합치고 싶을 경우
# df['최종 태그'] = df['태그'] + ", " + df['본문 내 태그']

# 4. 정제된 결과 저장
df.to_csv('cleaned_naver_blog_reviews.csv', index=False, encoding='utf-8-sig')
print("본문과 태그 분리 및 정제가 완료되었습니다.")

본문과 태그 분리 및 정제가 완료되었습니다.
